# NDR - Notebook 41b: Local maCADD Demote Calibration Probe (gated, no-submit)

## Why (evidence, not guess)
- NB40: USE_TRIGGER=False (shared NC fail)
- NB41a: USE_DEMOTE=False (best AUC s_sim=0.695)
- Remaining measurable question: even with weak ROC, does soft demote calibrated on local maCADD A=10 beat pass-through safely?

## Gates
```
IMPROVE_OK : (BASE - BEST) / BASE >= 0.05
SAFE_OK    : clean conf ratio after/before >= 0.95
USE_CALIB  = IMPROVE_OK and SAFE_OK
```
True -> A/B vs 229 only. False -> keep neo 229.


In [1]:
!pip install -q setuptools
!pip install -q git+https://github.com/facebookresearch/detectron2.git


  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 7.4 MB/s eta 0:00:00


In [2]:
import copy, json, math, os, random, sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from detectron2 import model_zoo
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.config import get_cfg
from detectron2.data import (
    DatasetCatalog, DatasetMapper, MetadataCatalog,
    build_detection_train_loader, detection_utils as utils,
)
from detectron2.engine import DefaultPredictor, DefaultTrainer
from detectron2.modeling import build_model
from scipy.optimize import linear_sum_assignment
import logging
logging.getLogger("detectron2").setLevel(logging.ERROR)
logging.getLogger("fvcore").setLevel(logging.ERROR)

print("python", sys.version.split()[0])
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), "ngpu", torch.cuda.device_count())

CANDIDATES = []
kaggle_input = Path("/kaggle/input")
if kaggle_input.exists():
    print("=== /kaggle/input listing ===")
    for pth in sorted(kaggle_input.iterdir()):
        print(" ", pth)
        CANDIDATES.append(pth)
        try:
            for q in pth.iterdir():
                if q.is_dir():
                    CANDIDATES.append(q)
        except Exception:
            pass
CANDIDATES.extend([
    Path("/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models"),
    Path("/kaggle/input/neural-debris-removal-in-streak-detection-models"),
    Path(r"D:/Code/Kaggle/Neural Debris Removal in Streak Detection Models/data"),
])

def looks_like_root(c):
    if not c.exists() or not c.is_dir():
        return False
    w1 = c / "poisoned_model" / "poisoned_model.pth"
    w2 = c / "poisoned_model.pth"
    u = c / "unlearn_set"
    return (w1.exists() or w2.exists()) and u.exists()

ROOT = None
for c in CANDIDATES:
    try:
        if looks_like_root(c):
            ROOT = c
            break
    except Exception:
        pass
if ROOT is None and kaggle_input.exists():
    for wpath in kaggle_input.rglob("poisoned_model.pth"):
        cand = wpath.parent.parent if wpath.parent.name == "poisoned_model" else wpath.parent
        if looks_like_root(cand):
            ROOT = cand
            break
if ROOT is None:
    raise FileNotFoundError("Cannot find competition data root.")

if (ROOT / "poisoned_model" / "poisoned_model.pth").exists():
    POISONED_WEIGHTS = str(ROOT / "poisoned_model" / "poisoned_model.pth")
else:
    POISONED_WEIGHTS = str(ROOT / "poisoned_model.pth")
UNLEARN_DIR = ROOT / "unlearn_set"
TEST_DIR = ROOT / "test_set" / "test_set"
if not TEST_DIR.exists():
    TEST_DIR = ROOT / "test_set"
if not TEST_DIR.exists():
    TEST_DIR = ROOT / "test"
if Path("/kaggle/working").exists():
    WORK = Path("/kaggle/working")
else:
    WORK = Path(r"D:/Code/Kaggle/Neural Debris Removal in Streak Detection Models/_probe41b")
WORK.mkdir(parents=True, exist_ok=True)
OUT_FT = WORK / "cls_only_ft"
OUT_FT.mkdir(parents=True, exist_ok=True)

BASE_CONFIG = "COCO-Detection/retinanet_R_50_FPN_3x.yaml"
ANCHOR_ASPECT_RATIOS = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
ANCHOR_SIZES = [[16], [32], [64], [128], [256]]
NUM_CLASSES = 1
IMG_W = IMG_H = 1024
CAND_THRESH = 0.05
MATCH_IOU = 0.50
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

N_BG = 20 if DEVICE == "cuda" else 10
N_POISON = 40 if DEVICE == "cuda" else 20
N_CLEAN = 40 if DEVICE == "cuda" else 20
FT_ITERS = 20 if DEVICE == "cuda" else 5
FT_LR = 5e-4
CROP = 64
A_FACTOR = 10.0
IOU_THRESHOLDS = np.arange(0.2, 0.91, 0.1)
CLEAN_CONF_FLOOR = 0.20

print("ROOT=", ROOT)
print("device=", DEVICE)
print("weights exists", Path(POISONED_WEIGHTS).exists())
print("WORK=", WORK)
print("budget: N_BG=%d N_POISON=%d N_CLEAN=%d FT_ITERS=%d" % (N_BG, N_POISON, N_CLEAN, FT_ITERS))


python 3.12.13
torch 2.10.0+cu128 cuda True ngpu 2
=== /kaggle/input listing ===
  /kaggle/input/competitions
  /kaggle/input/datasets
ROOT= /kaggle/input/competitions/neural-debris-removal-in-streak-detection-models
device= cuda
weights exists True
WORK= /kaggle/working
budget: N_BG=20 N_POISON=40 N_CLEAN=40 FT_ITERS=20


In [3]:
def load_image(path):
    im = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if im is None:
        raise FileNotFoundError(path)
    if im.dtype == np.uint16:
        im = im.astype(np.float32) / 65535.0
    im = np.clip(im * 255, 0, 255).astype(np.float32)
    if im.ndim == 2:
        im = np.repeat(im[:, :, None], 3, axis=2)
    return im

def build_cfg(weights=POISONED_WEIGHTS, score_thresh=CAND_THRESH, output_dir=None):
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file(BASE_CONFIG))
    cfg.MODEL.WEIGHTS = weights
    cfg.MODEL.RETINANET.NUM_CLASSES = NUM_CLASSES
    cfg.MODEL.RETINANET.SCORE_THRESH_TEST = score_thresh
    cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [ANCHOR_ASPECT_RATIOS]
    cfg.MODEL.ANCHOR_GENERATOR.SIZES = ANCHOR_SIZES
    cfg.MODEL.DEVICE = DEVICE
    cfg.INPUT.FORMAT = "RGB"
    if output_dir is not None:
        cfg.OUTPUT_DIR = str(output_dir)
        Path(output_dir).mkdir(parents=True, exist_ok=True)
    return cfg

def build_predictor(weights=POISONED_WEIGHTS, score_thresh=CAND_THRESH):
    return DefaultPredictor(build_cfg(weights, score_thresh))

def predict_image(predictor, img):
    inst = predictor(img)["instances"].to("cpu")
    boxes = inst.pred_boxes.tensor.numpy().astype(np.float32)
    scores = inst.scores.numpy().astype(np.float32)
    return boxes, scores

def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), np.float32)
    ax1, ay1, ax2, ay2 = a[:, 0:1], a[:, 1:2], a[:, 2:3], a[:, 3:4]
    bx1, by1, bx2, by2 = b[:, 0], b[:, 1], b[:, 2], b[:, 3]
    iw = np.maximum(0, np.minimum(ax2, bx2) - np.maximum(ax1, bx1))
    ih = np.maximum(0, np.minimum(ay2, by2) - np.maximum(ay1, by1))
    inter = iw * ih
    aa = np.maximum(0, ax2 - ax1) * np.maximum(0, ay2 - ay1)
    bb = np.maximum(0, bx2 - bx1) * np.maximum(0, by2 - by1)
    return (inter / np.maximum(aa + bb - inter, 1e-6)).astype(np.float32)

with open(UNLEARN_DIR / "annotations_coco.json") as f:
    coco = json.load(f)
id_to_file = {im["id"]: im["file_name"] for im in coco["images"]}
stem_by_id = {im["id"]: Path(im["file_name"]).stem for im in coco["images"]}

poison_wh = []
for ann in coco["annotations"]:
    x, y, w, h = ann["bbox"]
    poison_wh.append([w, h])
poison_wh = np.array(poison_wh, np.float32)

unlearn_files = sorted(Path(UNLEARN_DIR).glob("*.png"))
test_files = sorted(Path(TEST_DIR).glob("*.png"))
print("n unlearn", len(unlearn_files), "n test", len(test_files), "n anns", len(coco["annotations"]))
poisoned_predictor = build_predictor(POISONED_WEIGHTS, CAND_THRESH)
print("poisoned predictor ready")


n unlearn 20 n test 2000 n anns 20
poisoned predictor ready


In [4]:
MARGIN = 6

def extract_patch(img, xyxy, margin=MARGIN):
    x1, y1, x2, y2 = [int(round(v)) for v in xyxy]
    x1 = max(0, x1 - margin); y1 = max(0, y1 - margin)
    x2 = min(IMG_W, x2 + margin); y2 = min(IMG_H, y2 + margin)
    if x2 - x1 < 8 or y2 - y1 < 8:
        return None
    return img[y1:y2, x1:x2].copy(), (x1, y1, x2, y2)

poison_patches = []
missing = 0
for ann in coco["annotations"]:
    iid = ann["image_id"]
    fn = id_to_file[iid]
    path = UNLEARN_DIR / fn
    if not path.exists():
        cands = list(UNLEARN_DIR.rglob(Path(fn).name))
        if not cands:
            missing += 1
            continue
        path = cands[0]
    img = load_image(path)
    x, y, w, h = ann["bbox"]
    xyxy = [x, y, x + w, y + h]
    out = extract_patch(img, xyxy)
    if out is None:
        missing += 1
        continue
    patch, _ = out
    poison_patches.append({"patch": patch.astype(np.float32), "xyxy": xyxy, "stem": stem_by_id[iid]})

print("poison patches: %d missing=%d" % (len(poison_patches), missing))
assert len(poison_patches) >= 10

rng = np.random.default_rng(SEED)
n_pick = min(40, len(test_files))
idxs = rng.choice(len(test_files), size=n_pick, replace=False)
scored = []
for k in idxs:
    pth = test_files[int(k)]
    img = load_image(pth)
    with torch.no_grad():
        b, s = predict_image(poisoned_predictor, img)
    n_hi = int((s >= 0.2).sum()) if len(s) else 0
    scored.append((img, n_hi, pth.stem))
scored.sort(key=lambda t: t[1])
bgs = [im for im, _, _ in scored[:N_BG]]
print("backgrounds selected:", len(bgs), "det_counts", [n for _, n, _ in scored[:N_BG]])


poison patches: 20 missing=0
backgrounds selected: 20 det_counts [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1]


In [5]:
def paste_patch(img, patch, x0, y0, feather=2):
    out = img.astype(np.float32, copy=True)
    patch = patch.astype(np.float32, copy=False)
    ph, pw = patch.shape[:2]
    if y0 < 0 or x0 < 0 or y0 + ph > IMG_H or x0 + pw > IMG_W:
        return None
    if feather > 0:
        yy = np.linspace(0, 1, ph, dtype=np.float32)[:, None]
        xx = np.linspace(0, 1, pw, dtype=np.float32)[None, :]
        edge = np.minimum(np.minimum(yy, 1.0 - yy), np.minimum(xx, 1.0 - xx))
        alpha = np.clip(edge / max(feather / max(ph, pw), 1e-3), 0, 1).astype(np.float32)[..., None]
        region = out[y0:y0+ph, x0:x0+pw]
        if patch.ndim == 2:
            patch = np.repeat(patch[:, :, None], region.shape[2], axis=2)
        if patch.shape[2] != region.shape[2]:
            patch = np.repeat(patch[:, :, :1], region.shape[2], axis=2)
        out[y0:y0+ph, x0:x0+pw] = region * (1.0 - alpha) + patch * alpha
    else:
        if patch.ndim == 2:
            patch = np.repeat(patch[:, :, None], out.shape[2], axis=2)
        out[y0:y0+ph, x0:x0+pw] = patch
    return out

def continuous_streak_patch(pw, ph, rng, bright_scale=None, bg_level=None):
    canvas = np.zeros((ph, pw), np.float32)
    ang = rng.uniform(0, np.pi)
    L = 0.9 * min(pw, ph)
    cx, cy = pw / 2, ph / 2
    p1 = (int(cx - L/2 * np.cos(ang)), int(cy - L/2 * np.sin(ang)))
    p2 = (int(cx + L/2 * np.cos(ang)), int(cy + L/2 * np.sin(ang)))
    thick = max(1, int(round(min(pw, ph) * 0.08)))
    cv2.line(canvas, p1, p2, 1.0, thick)
    canvas = cv2.GaussianBlur(canvas, (0, 0), 0.8)
    if bright_scale is None:
        bright_scale = 40.0
    if bg_level is None:
        bg_level = 20.0
    g = bg_level + canvas / (canvas.max() + 1e-6) * bright_scale
    g = np.clip(g, 0, 255).astype(np.float32)
    return np.repeat(g[:, :, None], 3, axis=2)

def rand_free_spot(bg, pw, ph, rng, max_tries=40):
    for _ in range(max_tries):
        if IMG_W - pw <= 1 or IMG_H - ph <= 1:
            return None
        x0 = int(rng.integers(0, IMG_W - pw))
        y0 = int(rng.integers(0, IMG_H - ph))
        return x0, y0
    return None

p90 = float(np.percentile(np.concatenate([p["patch"].reshape(-1) for p in poison_patches]), 90))
p20 = float(np.percentile(np.concatenate([p["patch"].reshape(-1) for p in poison_patches]), 20))
print("poison intensity p20/p90", p20, p90)


poison intensity p20/p90 32.583656311035156 64.6926040649414


In [6]:
def match_fire(boxes, scores, xyxy, conf=0.15, iou_thr=0.10):
    if len(boxes) == 0:
        return -1, 0.0
    tx1, ty1, tx2, ty2 = xyxy
    region = np.array([[tx1, ty1, tx2, ty2]], np.float32)
    ious = iou_matrix(boxes, region)[:, 0]
    cx = 0.5 * (boxes[:, 0] + boxes[:, 2])
    cy = 0.5 * (boxes[:, 1] + boxes[:, 3])
    inside = (cx >= tx1) & (cx <= tx2) & (cy >= ty1) & (cy <= ty2)
    score = ious.copy()
    score[inside] = np.maximum(score[inside], 0.15)
    order = np.argsort(-score)
    for j in order:
        if scores[j] < conf:
            continue
        if score[j] >= iou_thr or inside[j]:
            return int(j), float(scores[j])
    return -1, 0.0

def build_labeled_cases(n_poison, n_clean, seed=SEED):
    rng = np.random.default_rng(seed)
    cases = []
    tries = 0
    while len([c for c in cases if c["label"] == 1]) < n_poison and tries < n_poison * 8:
        tries += 1
        pp = poison_patches[int(rng.integers(len(poison_patches)))]
        patch = pp["patch"]; ph, pw = patch.shape[:2]
        bg = bgs[int(rng.integers(len(bgs)))]
        spot = rand_free_spot(bg, pw, ph, rng)
        if spot is None:
            continue
        x0, y0 = spot
        im2 = paste_patch(bg, patch, x0, y0)
        if im2 is None:
            continue
        xyxy = [x0 + 2, y0 + 2, x0 + pw - 2, y0 + ph - 2]
        boxes, scores = predict_image(poisoned_predictor, im2)
        j, c = match_fire(boxes, scores, xyxy)
        if j < 0:
            continue
        cases.append({"label": 1, "kind": "poison_real", "image": im2, "xyxy": xyxy,
                      "cand_box": boxes[j].astype(np.float32), "cand_score": float(scores[j])})
    tries = 0
    while len([c for c in cases if c["label"] == 0]) < n_clean and tries < n_clean * 10:
        tries += 1
        wh = poison_wh[int(rng.integers(len(poison_wh)))]
        pw = int(np.clip(wh[0] * rng.uniform(0.85, 1.15), 16, 120))
        ph = int(np.clip(wh[1] * rng.uniform(0.85, 1.15), 12, 120))
        if pw % 2: pw += 1
        if ph % 2: ph += 1
        patch = continuous_streak_patch(pw, ph, rng, bright_scale=max(20.0, p90 - p20), bg_level=p20)
        bg = bgs[int(rng.integers(len(bgs)))]
        spot = rand_free_spot(bg, pw, ph, rng)
        if spot is None:
            continue
        x0, y0 = spot
        im2 = paste_patch(bg, patch, x0, y0)
        if im2 is None:
            continue
        xyxy = [x0 + 2, y0 + 2, x0 + pw - 2, y0 + ph - 2]
        boxes, scores = predict_image(poisoned_predictor, im2)
        j, c = match_fire(boxes, scores, xyxy)
        if j < 0:
            continue
        cases.append({"label": 0, "kind": "clean_continuous", "image": im2, "xyxy": xyxy,
                      "cand_box": boxes[j].astype(np.float32), "cand_score": float(scores[j])})
    return cases

print("Building labeled transplant detections ...")
cases = build_labeled_cases(N_POISON, N_CLEAN, seed=SEED)
n_pos = sum(c["label"] == 1 for c in cases)
n_neg = sum(c["label"] == 0 for c in cases)
print("cases total=%d poison=%d clean=%d" % (len(cases), n_pos, n_neg))
print("poison cand_score mean=%.3f | clean cand_score mean=%.3f" % (
    np.mean([c["cand_score"] for c in cases if c["label"]==1] or [0]),
    np.mean([c["cand_score"] for c in cases if c["label"]==0] or [0]),
))
assert n_pos >= 10 and n_neg >= 10


Building labeled transplant detections ...
cases total=80 poison=40 clean=40
poison cand_score mean=0.608 | clean cand_score mean=0.450


In [7]:
UNLEARN_DATASET = "unlearn_nb41b"

def register_unlearn():
    if UNLEARN_DATASET in DatasetCatalog:
        DatasetCatalog.remove(UNLEARN_DATASET)
    dicts = []
    for im in coco["images"]:
        fp = UNLEARN_DIR / im["file_name"]
        if not fp.exists():
            cands = list(UNLEARN_DIR.rglob(Path(im["file_name"]).name))
            fp = cands[0] if cands else fp
        dicts.append({
            "file_name": str(fp),
            "image_id": im["id"],
            "height": im.get("height", IMG_H),
            "width": im.get("width", IMG_W),
            "annotations": [],
        })
    DatasetCatalog.register(UNLEARN_DATASET, lambda d=dicts: d)
    MetadataCatalog.get(UNLEARN_DATASET).set(thing_classes=["object"])
    return dicts

class EmptyLabelMapper(DatasetMapper):
    def __call__(self, dataset_dict):
        dataset_dict = copy.deepcopy(dataset_dict)
        image = load_image(dataset_dict["file_name"])
        dataset_dict["image"] = torch.as_tensor(image.transpose(2, 0, 1).copy())
        dataset_dict["instances"] = utils.annotations_to_instances([], image.shape[:2])
        return dataset_dict

class ClsOnlyTrainer(DefaultTrainer):
    @classmethod
    def build_train_loader(cls, cfg):
        # pass dataset dicts directly so Detectron2 does NOT filter empty anns
        mapper = EmptyLabelMapper(cfg, is_train=True, augmentations=[])
        return build_detection_train_loader(
            cfg, mapper=mapper, dataset=DatasetCatalog.get(cfg.DATASETS.TRAIN[0])
        )

    @classmethod
    def build_model(cls, cfg):
        model = super().build_model(cfg)
        for name, p in model.named_parameters():
            p.requires_grad = ("cls_score" in name) or ("cls_subnet" in name)
        n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
        n_tot = sum(p.numel() for p in model.parameters())
        pct = 100.0 * n_tr / max(n_tot, 1)
        print("cls-only trainable", n_tr, "/", n_tot, "pct", round(pct, 1))
        return model

FT_WEIGHTS = OUT_FT / "model_final.pth"
if FT_WEIGHTS.exists():
    print("reuse cached FT weights", FT_WEIGHTS)
else:
    print("Training cls-only empty-label FT: lr=%s iters=%d" % (FT_LR, FT_ITERS))
    register_unlearn()
    cfg_ft = build_cfg(POISONED_WEIGHTS, score_thresh=CAND_THRESH, output_dir=OUT_FT)
    cfg_ft.DATASETS.TRAIN = (UNLEARN_DATASET,)
    cfg_ft.DATASETS.TEST = ()
    cfg_ft.DATALOADER.FILTER_EMPTY_ANNOTATIONS = False
    cfg_ft.SOLVER.IMS_PER_BATCH = 4
    cfg_ft.SOLVER.BASE_LR = FT_LR
    cfg_ft.SOLVER.MAX_ITER = FT_ITERS
    cfg_ft.SOLVER.STEPS = []
    cfg_ft.SOLVER.WARMUP_ITERS = 0
    cfg_ft.SOLVER.CHECKPOINT_PERIOD = 100000
    cfg_ft.DATALOADER.NUM_WORKERS = 0
    cfg_ft.MODEL.WEIGHTS = POISONED_WEIGHTS
    trainer = ClsOnlyTrainer(cfg_ft)
    trainer.resume_or_load(resume=False)
    trainer.train()
    print("FT done", FT_WEIGHTS, "exists", FT_WEIGHTS.exists())

if not FT_WEIGHTS.exists():
    cands = list(OUT_FT.rglob("model_final.pth"))
    if cands:
        FT_WEIGHTS = cands[0]
        print("found FT weights at", FT_WEIGHTS)
    else:
        raise FileNotFoundError("FT weights missing")

import gc
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

depoisoned_predictor = build_predictor(str(FT_WEIGHTS), CAND_THRESH)
print("depoisoned predictor ready from", FT_WEIGHTS)



Training cls-only empty-label FT: lr=0.0005 iters=20
[07/19 20:29:57 d2.engine.defaults]: Model:
RetinaNet(
  (backbone): FPN(
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelP6P7(
      (p6): Conv2d(2048, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (p7): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    )
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBatchNorm2d(num_f

2026-07-19 20:30:13.639730: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1784493013.834330      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1784493013.896156      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1784493014.383451      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784493014.383474      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784493014.383477      22 computation_placer.cc:177] computation placer alr

[07/19 20:30:25 d2.engine.hooks]: Overall training speed: 18 iterations in 0:00:11 (0.6262 s / it)
[07/19 20:30:25 d2.engine.hooks]: Total training time: 0:00:26 (0:00:15 on hooks)
FT done /kaggle/working/cls_only_ft/model_final.pth exists True
[07/19 20:30:26 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /kaggle/working/cls_only_ft/model_final.pth ...
depoisoned predictor ready from /kaggle/working/cls_only_ft/model_final.pth


In [8]:
def norm_crop_gray(img, box, size=CROP):
    x1, y1, x2, y2 = [float(v) for v in box]
    cx = 0.5 * (x1 + x2); cy = 0.5 * (y1 + y2)
    side = max(x2 - x1, y2 - y1) * 1.2
    half = side / 2.0
    a = int(np.clip(cy - half, 0, IMG_H)); b = int(np.clip(cy + half, 0, IMG_H))
    c = int(np.clip(cx - half, 0, IMG_W)); d = int(np.clip(cx + half, 0, IMG_W))
    g = img[a:b, c:d, 0] if img.ndim == 3 else img[a:b, c:d]
    if g.size == 0:
        g = np.zeros((8, 8), np.float32)
    g = cv2.resize(g.astype(np.float32), (size, size), interpolation=cv2.INTER_LINEAR)
    med = np.median(g); mad = np.median(np.abs(g - med)) + 1e-3
    return ((g - med) / (5 * mad)).astype(np.float32)

poison_bank = []
for pp in poison_patches:
    g = pp["patch"][:, :, 0]
    g = cv2.resize(g.astype(np.float32), (CROP, CROP), interpolation=cv2.INTER_LINEAR)
    med = np.median(g); mad = np.median(np.abs(g - med)) + 1e-3
    poison_bank.append(((g - med) / (5 * mad)).astype(np.float32))
poison_bank = np.stack(poison_bank, 0)
poison_bank_flat = poison_bank.reshape(len(poison_bank), -1)
poison_bank_flat = poison_bank_flat / (np.linalg.norm(poison_bank_flat, axis=1, keepdims=True) + 1e-6)

def sim_to_poison(img, box):
    crop = norm_crop_gray(img, box)
    v = crop.reshape(-1)
    v = v / (np.linalg.norm(v) + 1e-6)
    return float((poison_bank_flat @ v).max())

logwh = np.log(np.maximum(poison_wh, 1e-3))
MU = logwh.mean(0)
COV = np.cov(logwh.T) + 1e-3 * np.eye(2)
COV_INV = np.linalg.inv(COV)

def geometry_score_box(box):
    w = max(float(box[2] - box[0]), 1e-3)
    h = max(float(box[3] - box[1]), 1e-3)
    d = np.log([w, h]) - MU
    m2 = float(d @ COV_INV @ d)
    return float(np.exp(-0.5 * m2 / 2.0))

def s_diff_for_case(case):
    boxes_d, scores_d = predict_image(depoisoned_predictor, case["image"])
    cb = case["cand_box"][None, :]
    cs = case["cand_score"]
    if len(boxes_d) == 0:
        return 1.0
    ious = iou_matrix(cb, boxes_d)[0]
    j = int(ious.argmax())
    if ious[j] < MATCH_IOU:
        return 1.0
    return float(1.0 - np.clip(scores_d[j] / max(cs, 1e-6), 0, 1))

rows = []
for i, case in enumerate(cases):
    sd = s_diff_for_case(case)
    smax = sim_to_poison(case["image"], case["cand_box"])
    sg = geometry_score_box(case["cand_box"])
    rows.append({
        "idx": i, "label": int(case["label"]), "cand_score": case["cand_score"],
        "s_diff": sd, "s_sim": smax, "s_geo": sg,
        "blend_229": 0.90 * sd + 0.10 * sg,
        "blend_simdiff": 0.50 * sd + 0.50 * smax,
    })
    if (i + 1) % 20 == 0:
        print("scored", i + 1, "/", len(cases))

df = pd.DataFrame(rows)
print(df.groupby("label")[["s_diff", "s_sim", "s_geo", "blend_229", "blend_simdiff"]].median())
df.to_csv(WORK / "probe41b_scores.csv", index=False)
print("saved", WORK / "probe41b_scores.csv")


scored 20 / 80
scored 40 / 80
scored 60 / 80
scored 80 / 80
         s_diff     s_sim     s_geo  blend_229  blend_simdiff
label                                                        
0      0.768604  0.117887  0.605931   0.757103       0.470078
1      0.799076  0.307206  0.672483   0.779032       0.531775
saved /kaggle/working/probe41b_scores.csv


In [9]:
def acadd_t(cb, cs, pb, ps, t, cip):
    ious = iou_matrix(cb, pb)
    cost = 0.0
    mc = set(); mp = set()
    if ious.size:
        ri, ci = linear_sum_assignment(-ious)
        for i, j in zip(ri, ci):
            if ious[i, j] >= t:
                mc.add(i); mp.add(j)
                d = float(cs[i]) - float(ps[j])
                if cip[i]:
                    cost += (d / A_FACTOR) if d > 0 else -d
                else:
                    cost += d if d > 0 else (-d / A_FACTOR)
    cost += sum(float(cs[i]) for i in range(len(cs)) if i not in mc)
    cost += sum(float(ps[j]) for j in range(len(ps)) if j not in mp)
    return cost

def macadd_one(cb, cs, pb, ps, cip):
    w = IOU_THRESHOLDS / IOU_THRESHOLDS.sum()
    keep = cs > CLEAN_CONF_FLOOR
    cb2, cs2 = cb[keep], cs[keep]
    cip2 = cip[keep]
    if len(cs2) == 0 and len(ps) == 0:
        return 0.0
    if len(cs2) == 0:
        return float(ps.sum())
    return float(sum(wi * acadd_t(cb2, cs2, pb, ps, t, cip2) for wi, t in zip(w, IOU_THRESHOLDS)))

def remap_confidence(s, p, R):
    if s < R["MIN_KEEP"]:
        return 0.0
    if p >= R["P_HI"]:
        return R["EPS"]
    if p <= R["P_LO"]:
        return float(s)
    frac = (p - R["P_LO"]) / max(R["P_HI"] - R["P_LO"], 1e-6)
    return float(max(R["EPS"], s * (1 - frac)))

def eval_policy(score_col, R, name):
    costs = []
    clean_before = []; clean_after = []
    poison_before = []; poison_after = []
    for i, case in enumerate(cases):
        row = df.iloc[i]
        p = float(row[score_col])
        s0 = float(case["cand_score"])
        s1 = remap_confidence(s0, p, R)
        cb = case["cand_box"][None, :].astype(np.float32)
        cs = np.array([s0], np.float32)
        cip = np.array([case["label"] == 1], bool)
        pb = cb.copy()
        ps = np.array([s1], np.float32)
        costs.append(macadd_one(cb, cs, pb, ps, cip))
        if case["label"] == 0:
            clean_before.append(s0); clean_after.append(s1)
        else:
            poison_before.append(s0); poison_after.append(s1)
    return {
        "name": name, "score_col": score_col,
        "macadd_mean": float(np.mean(costs)),
        "clean_conf_ratio": float(np.mean(clean_after) / max(np.mean(clean_before), 1e-6)),
        "poison_conf_ratio": float(np.mean(poison_after) / max(np.mean(poison_before), 1e-6)),
        "clean_mean_after": float(np.mean(clean_after)),
        "poison_mean_after": float(np.mean(poison_after)),
    }

R_base = {"MIN_KEEP": 0.0, "P_HI": 2.0, "P_LO": 1.0, "EPS": 0.01}
base = eval_policy("s_sim", R_base, "baseline_passthrough")
print("BASE macadd_mean=%.4f clean_ratio=%.3f poison_ratio=%.3f" % (
    base["macadd_mean"], base["clean_conf_ratio"], base["poison_conf_ratio"]))

grid = []
for score_col in ["s_sim", "blend_229", "blend_simdiff", "s_diff"]:
    for p_lo, p_hi, eps, min_keep in [
        (0.25, 0.55, 0.01, 0.20),
        (0.30, 0.60, 0.01, 0.20),
        (0.35, 0.65, 0.01, 0.20),
        (0.40, 0.70, 0.01, 0.15),
        (0.20, 0.50, 0.05, 0.20),
        (0.45, 0.75, 0.01, 0.20),
    ]:
        R = {"MIN_KEEP": min_keep, "P_HI": p_hi, "P_LO": p_lo, "EPS": eps}
        name = "%s_lo%.2f_hi%.2f_mk%.2f" % (score_col, p_lo, p_hi, min_keep)
        grid.append(eval_policy(score_col, R, name))

gdf = pd.DataFrame(grid).sort_values("macadd_mean")
print("=== TOP 10 DEMOTE POLICIES (lower maCADD better) ===")
print(gdf[["name", "macadd_mean", "clean_conf_ratio", "poison_conf_ratio",
           "clean_mean_after", "poison_mean_after"]].head(10).to_string(index=False))
gdf.to_csv(WORK / "probe41b_grid.csv", index=False)

best = gdf.iloc[0].to_dict()
BASE_M = base["macadd_mean"]
BEST_M = float(best["macadd_mean"])
rel_improve = (BASE_M - BEST_M) / max(BASE_M, 1e-6)
IMPROVE_OK = rel_improve >= 0.05
SAFE_OK = float(best["clean_conf_ratio"]) >= 0.95
USE_CALIB = bool(IMPROVE_OK and SAFE_OK)

print("=" * 64)
print("NB41b LOCAL maCADD CALIB - GATES")
print("=" * 64)
print("BASE macadd_mean = %.4f" % BASE_M)
print("BEST macadd_mean = %.4f  policy=%s" % (BEST_M, best["name"]))
print("REL_IMPROVE      = %.3f   (need >= 0.05)  IMPROVE_OK=%s" % (rel_improve, IMPROVE_OK))
print("CLEAN_RATIO      = %.3f   (need >= 0.95)  SAFE_OK=%s" % (float(best["clean_conf_ratio"]), SAFE_OK))
print("POISON_RATIO     = %.3f" % float(best["poison_conf_ratio"]))
print("USE_CALIB        = %s" % USE_CALIB)
print("-" * 64)
if USE_CALIB:
    print("PASS: local maCADD improves with safe clean retention.")
    print("Next: A/B submit this demote on top of 229 weights only.")
else:
    print("FAIL: soft demote on measured signals does not beat pass-through safely.")
    print("Keep neo 229. Do not open new demote notebook.")
print("=" * 64)

decision = {
    "base_macadd_mean": BASE_M,
    "best_macadd_mean": BEST_M,
    "best_policy": best["name"],
    "rel_improve": rel_improve,
    "clean_conf_ratio": float(best["clean_conf_ratio"]),
    "poison_conf_ratio": float(best["poison_conf_ratio"]),
    "IMPROVE_OK": IMPROVE_OK,
    "SAFE_OK": SAFE_OK,
    "USE_CALIB": USE_CALIB,
    "n_poison": int(n_pos),
    "n_clean": int(n_neg),
}
with open(WORK / "probe41b_decision.json", "w") as f:
    json.dump(decision, f, indent=2)
print("saved", WORK / "probe41b_decision.json")
print("USE_CALIB=", USE_CALIB, "best=", best["name"])


BASE macadd_mean=0.0157 clean_ratio=1.000 poison_ratio=1.000
=== TOP 10 DEMOTE POLICIES (lower maCADD better) ===
                              name  macadd_mean  clean_conf_ratio  poison_conf_ratio  clean_mean_after  poison_mean_after
        s_sim_lo0.45_hi0.75_mk0.20     0.002602          0.939685           0.907434          0.423309           0.551432
        s_sim_lo0.35_hi0.65_mk0.20     0.005941          0.938646           0.805223          0.422841           0.489320
        s_sim_lo0.30_hi0.60_mk0.20     0.014214          0.910904           0.738599          0.410344           0.448833
        s_sim_lo0.40_hi0.70_mk0.15     0.019806          1.000000           0.864609          0.450480           0.525408
        s_sim_lo0.25_hi0.55_mk0.20     0.028503          0.858810           0.654502          0.386877           0.397729
blend_simdiff_lo0.45_hi0.75_mk0.20     0.039859          0.806922           0.665409          0.363502           0.404357
        s_sim_lo0.20_hi0.50_mk0.

## How to run
1. Upload to Kaggle (GPU T4) + competition data
2. Run All
3. Paste: TOP 10 table + USE_CALIB / IMPROVE_OK / SAFE_OK
